# Kion data split

Disjoint global timestamp split: **80% train / 10% validation / 10% test**.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
from preprocess import train_val_test_split, prepare_splitted_data

PROJECT_PATH = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_PATH, 'data', 'kion_genres')
os.makedirs(DATA_PATH, exist_ok=True)

RELEVANCE_COL = 'watched_pct'
RELEVANCE_THRESHOLD = 15

In [ ]:
RAW_PATH = os.environ.get(
    'KION_INTERACTIONS_PATH',
    os.path.join(PROJECT_PATH, 'data', 'interactions_df.csv'),
)
ITEMS_PATH = os.environ.get(
    'KION_ITEMS_PATH',
    os.path.join(PROJECT_PATH, 'data', 'items.csv'),
)

data = pd.read_csv(RAW_PATH)
data = data.rename(columns={'last_watch_dt': 'timestamp'})
data = data[data.total_dur > 300]
data = data.sort_values(['user_id', 'timestamp'])

# Optional subsample for faster experimentation
SUBSAMPLE_USERS = 50_000
if SUBSAMPLE_USERS and data.user_id.nunique() > SUBSAMPLE_USERS:
    random_user_ids = data['user_id'].sample(n=SUBSAMPLE_USERS, random_state=42)
    data = data[data.user_id.isin(random_user_ids)]

item_dict = dict(zip(data['item_id'], data['item_id'].astype('category').cat.codes))
data.head()

In [ ]:
print('users:', data.user_id.nunique())
print('items:', data.item_id.nunique())
print('negative feedback rate:', np.mean(data[RELEVANCE_COL] < RELEVANCE_THRESHOLD))

In [ ]:
train, val, test = train_val_test_split(
    data,
    RELEVANCE_THRESHOLD,
    RELEVANCE_COL,
    train_quantile=0.8,
    val_quantile=0.9,
)

In [ ]:
print('train users:', train.user_id.nunique())
print('val users:', val.user_id.nunique())
print('test users:', test.user_id.nunique())
train.groupby('user_id')['item_id'].count().describe()

In [ ]:
train.to_parquet(os.path.join(DATA_PATH, 'train.parquet'), index=False)
val.to_parquet(os.path.join(DATA_PATH, 'validation.parquet'), index=False)
test.to_parquet(os.path.join(DATA_PATH, 'test.parquet'), index=False)

items = pd.read_csv(ITEMS_PATH)
items['item_id'] = items['item_id'].map(item_dict)
items.to_parquet(os.path.join(DATA_PATH, 'items.parquet'), index=False)
print('saved to', DATA_PATH)

In [ ]:
(
    train_p,
    validation_p,
    test_p,
    last_pos_item_test,
    last_pos_item_val,
    last_neg_item_test,
    last_neg_item_val,
) = prepare_splitted_data(
    DATA_PATH,
    relevance_col=RELEVANCE_COL,
    relevance_threshold=RELEVANCE_THRESHOLD,
    verify=True,
)

print('val users with neighbour pair:', last_pos_item_val.user_id.nunique())
print('test users with neighbour pair:', last_pos_item_test.user_id.nunique())